In [28]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

In [3]:
df = pd.read_csv("../data/processed/games_march2025_cleaned.csv")
df = df.sort_values(by="recommendations", ascending=False)
df.head(1)

,appid,name,release_date,required_age,price,dlc_count,header_image,windows,mac,linux,...,positive,negative,average_playtime_forever,median_playtime_forever,discount,peak_ccu,pct_pos_total,num_reviews_total,tags_sep,publisher
0,2358720,Black Myth: Wukong,2024-08-19,13.0,59.99,2,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,1098373,37811,0,0,0,35990,96.0,825621.0,"['Mythology', 'Action RPG', 'Action', 'Souls-l...",Game Science


### Quanto o 1° se diferencia dos demais em preço?

In [4]:
prices = df[:10].price.values
prices

array([59.99, 39.99, 19.99, 29.99, 39.99, 38.99, 29.99,  4.49, 34.99,
       34.99])

In [5]:
idx_to_compare = 0
prices_sem_itc = np.delete(prices, idx_to_compare)

In [6]:
np.average(prices_sem_itc)

30.37888888888889

In [7]:
prices[idx_to_compare] / np.average(prices_sem_itc) # quase o dobro da média

1.974726601075308

In [8]:
idx_to_compare = 4
prices_sem_itc = np.delete(prices, idx_to_compare)
prices[idx_to_compare] / np.average(prices_sem_itc) # menor, mais ainda mais que a média

1.2266453086125215

In [9]:
idx_to_compare = -3
prices_sem_itc = np.delete(prices, idx_to_compare)
prices[idx_to_compare] / np.average(prices_sem_itc) # bem abaixo da média

0.12286035693654798

In [10]:
idx_to_compare = 0
colors = ['crimson' if i == idx_to_compare else 'lightblue' for i in range(len(prices))]
media_sem_idx0 = np.mean(np.delete(prices, idx_to_compare))
fig_jogo_preco = go.Figure()

fig_jogo_preco.add_trace(go.Bar(
    x=[i for i in df[:10].name.values],
    y=prices,
    marker_color=colors,
    text=[f"{p:.2f}" for p in prices],
    textposition="outside",
    name="Jogo Destacado"
))

fig_jogo_preco.add_trace(go.Scatter(
    x=[i for i in df[:10].name.values],
    y=[media_sem_idx0]*len(prices),
    mode='lines',
    line=dict(color='green', dash='dash'),
    name=f'Média (sem o {idx_to_compare+1}º jogo)'
))

fig_jogo_preco.update_layout(
    title=f'Gráfico de Preços com Destaque no {idx_to_compare+1}º Jogo',
    xaxis_title='Jogo',
    yaxis_title='Preço ($)',
    showlegend=True,
    bargap=0.2,
    template='plotly_white'
)

fig_jogo_preco.show()

### Quanto tempo a mais de jogo?

In [11]:
top10 = df[:10]
x = top10.recommendations.values
y = top10.peak_ccu.values
names = top10.name.values

In [12]:
idx_to_compare = 0
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='markers',
    marker=dict(size=10, color='lightblue'),
    text=names
))

fig.add_trace(go.Scatter(
    x=[x[idx_to_compare]],
    y=[y[idx_to_compare]],
    mode='markers+text',
    marker=dict(size=14, color='crimson', symbol='circle'),
    textposition='top center',
    text=names[idx_to_compare]
))

fig.update_layout(
    title='Relação atividade x recomendações',
    xaxis_title='Recommendations',
    yaxis_title='Peak_ccu',
    template='plotly_white'
)

fig.show()

### Suporte de platfromas

In [13]:
df['plataformas'] = df.apply(
    lambda row: '+'.join([
        plat for plat, available in zip(['Windows', 'Mac', 'Linux'], [row['windows'], row['mac'], row['linux']])
        if available
    ]), axis=1
)

df[["windows", "mac", "linux", "plataformas"]].head()

,windows,mac,linux,plataformas
0,True,False,False,Windows
1,True,False,False,Windows
2,True,True,True,Windows+Mac+Linux
3,True,False,False,Windows
4,True,False,False,Windows


In [14]:
combo_counts = df['plataformas'].value_counts().reset_index()

### Preço médio por publisher

In [15]:
df[["publishers", "publisher", "price"]].head()

,publishers,publisher,price
0,['Game Science'],Game Science,59.99
1,['PlayStation Publishing LLC'],PlayStation Publishing LLC,39.99
2,['Techland'],Techland,19.99
3,['Newnight'],Newnight,29.99
4,['Coffee Stain Publishing'],Coffee Stain Publishing,39.99


In [16]:
df_mean_price_publisher = df[["publisher", "price"]].groupby("publisher").mean().sort_values("price", ascending=False)[:10].reset_index()

In [17]:
df_mean_price_publisher

,publisher,price
0,Activision,69.99
1,GSC Game World (worldwide),59.99
2,Game Science,59.99
3,Gearbox Publishing,59.99
4,"NIS America, Inc.",59.99
5,Eversim,59.99
6,2K Games,59.99
7,2K,59.99
8,PlayStation Publishing LLC,56.24
9,Bandai Namco Entertainment,54.99


In [18]:
colors = ['lightblue' for i in range(len(df_mean_price_publisher))]

fig_mean_price_publisher = go.Figure()

fig_mean_price_publisher.add_trace(go.Bar(
    x=df_mean_price_publisher.publisher,
    y=df_mean_price_publisher.price,
    marker_color=colors,
    text=[f"{p:.2f}" for p in df_mean_price_publisher.price],
    textposition="outside",
))

fig_mean_price_publisher.show()

### Relação entre Percentual de avaliações positivas em relação ao total de análises, preço e idade requerida (acima de 0 - livre)

In [103]:
df_filtered_age = df.query("required_age > 0")
fig_dlc = go.Figure()

fig_dlc.add_trace(go.Scatter(
    x=df_filtered_age["pct_pos_total"],
    y=df_filtered_age["price"],
    mode='markers',
    marker=dict(
        size=df_filtered_age["required_age"],
        color=df_filtered_age["required_age"],
        colorscale='blues',
        showscale=True,
        colorbar=dict(title="Idade Requerida")
    ),
    text=df_filtered_age["name"],
    hovertemplate=(
        "<b>%{text}</b><br>" +
        "Preço: R$ %{y:.2f}<br>" +
        "Avaliações Positivas: %{x:.1f}%<br>" +
        "Idade Requerida: %{marker.color} anos"
    )
))

fig_dlc.update_layout(
    title="Relação entre Avaliações Positivas, Preço e Idade Requerida dos Jogos",
    xaxis_title="Percentual de Avaliações Positivas (%)",
    yaxis_title="Preço ($)",
    template="plotly_white",
    height=600
)

fig_dlc.show()

### Filtro de gênero

In [104]:
df.tags_sep

0       ['Mythology', 'Action RPG', 'Action', 'Souls-l...
1       ['Online Co-Op', 'PvE', 'Third-Person Shooter'...
2       ['Zombies', 'Survival Horror', 'Horror', 'Onli...
3       ['Survival', 'Open World', 'Multiplayer', 'Co-...
4       ['Base-Building', 'Automation', 'Open World', ...
                              ...                        
1760    ['Lovecraftian', 'Psychological Horror', 'Walk...
1759    ['Simulation', 'VR', 'Strategy', 'Casual', 'Fi...
1758    ['Early Access', 'Card Game', 'Management', 'S...
1757    ['Casual', 'RPG', 'Loot', 'Adventure', 'Point ...
3021    ['Racing', 'Casual', 'Simulation', 'Arcade', '...
Name: tags_sep, Length: 3022, dtype: object

In [121]:
df.query("'Mythology' in tags_sep")

,appid,name,release_date,required_age,price,dlc_count,header_image,windows,mac,linux,...,median_playtime_forever,discount,peak_ccu,pct_pos_total,num_reviews_total,tags_sep,publisher,plataformas,len_full_audio_languages,len_supported_languages


In [126]:
df[df.tags_sep.str.contains('Mythology')]

,appid,name,release_date,required_age,price,dlc_count,header_image,windows,mac,linux,...,median_playtime_forever,discount,peak_ccu,pct_pos_total,num_reviews_total,tags_sep,publisher,plataformas,len_full_audio_languages,len_supported_languages
0,2358720,Black Myth: Wukong,2024-08-19,13.0,59.99,2,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,0,0,35990,96.0,825621.0,"['Mythology', 'Action RPG', 'Action', 'Souls-l...",Game Science,Windows,3,13
16,1145350,Hades II,2024-05-06,0.0,29.99,1,https://shared.akamai.steamstatic.com/store_it...,True,True,False,...,0,0,7793,94.0,57793.0,"['Action', 'Rogue-like', 'Rogue-lite', 'Hack a...",Supergiant Games,Windows+Mac,2,15
50,2322010,God of War Ragnarök,2024-09-19,17.0,59.99,3,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,0,0,2610,86.0,17998.0,"['Action', 'Adventure', 'Action-Adventure', 'R...",PlayStation Publishing LLC,Windows,14,22
59,1934680,Age of Mythology: Retold,2024-09-04,0.0,29.99,3,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,2641,0,6369,90.0,14207.0,"['Strategy', 'RTS', 'Base-Building', 'Mytholog...",Xbox Game Studios,Windows,12,25
118,2461850,Senua’s Saga: Hellblade II,2024-05-21,17.0,49.99,1,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,325,0,25,87.0,6051.0,"['Action-Adventure', 'Psychological Horror', '...",Xbox Game Studios,Windows,2,27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1685,2649800,Vinecard,2024-08-07,0.0,9.99,0,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,0,0,1,97.0,37.0,"['Early Access', 'Replay Value', 'Strategy', '...",IndieArk,Windows,3,4
1676,3437170,壶中日月 Timeless Teapot,2025-02-03,0.0,4.99,0,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,0,0,0,63.0,38.0,"['Strategy', 'Idler', 'Casual', 'Loot', 'Simul...",壶天云镜,Windows,1,1
1808,2146140,BlackForge: A Smithing Adventure,2024-06-13,0.0,9.99,0,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,0,50,1,61.0,31.0,"['Casual', 'Simulation', 'VR', 'Adventure', 'E...",Fast Travel Games,Windows,2,6
1798,1424420,Warchief,2024-06-27,0.0,24.99,0,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,0,0,0,62.0,32.0,"['Early Access', 'Action', 'Indie', 'Strategy'...",Honikou Games,Windows,3,29


In [115]:
series_genero = df.tags_sep.apply(lambda x: eval(x))

In [116]:
stack_genero = list()
for i in series_genero:
    stack_genero.extend(i)

In [117]:
stack_genero = set(stack_genero)

In [119]:
len(stack_genero)

431